## Simulation work for SGIMC

In [ ]:
import numpy as np


def get_metrics(real_score, predict_score):
    sorted_predict_score = np.array(
        sorted(list(set(np.array(predict_score).flatten())))
    )
    sorted_predict_score_num = len(sorted_predict_score)
    thresholds = sorted_predict_score[
        np.int32(sorted_predict_score_num * np.arange(1, 1000) / 1000)
    ]
    thresholds = np.mat(thresholds)
    thresholds_num = thresholds.shape[1]

    predict_score_matrix = np.tile(predict_score, (thresholds_num, 1))
    negative_index = np.where(predict_score_matrix < thresholds.T)
    positive_index = np.where(predict_score_matrix >= thresholds.T)
    predict_score_matrix[negative_index] = 0
    predict_score_matrix[positive_index] = 1
    TP = predict_score_matrix.dot(real_score.T)
    FP = predict_score_matrix.sum(axis=1) - TP
    FN = real_score.sum() - TP
    TN = len(real_score.T) - TP - FP - FN

    fpr = FP / (FP + TN)
    tpr = TP / (TP + FN)
    ROC_dot_matrix = np.mat(sorted(np.column_stack((fpr, tpr)).tolist())).T
    ROC_dot_matrix.T[0] = [0, 0]
    ROC_dot_matrix = np.c_[ROC_dot_matrix, [1, 1]]
    x_ROC = ROC_dot_matrix[0].T
    y_ROC = ROC_dot_matrix[1].T
    auc = 0.5 * (x_ROC[1:] - x_ROC[:-1]).T * (y_ROC[:-1] + y_ROC[1:])

    recall_list = tpr
    precision_list = TP / (TP + FP)
    PR_dot_matrix = np.mat(
        sorted(np.column_stack((recall_list, precision_list)).tolist())
    ).T
    PR_dot_matrix.T[0] = [0, 1]
    PR_dot_matrix = np.c_[PR_dot_matrix, [1, 0]]
    x_PR = PR_dot_matrix[0].T
    y_PR = PR_dot_matrix[1].T
    aupr = 0.5 * (x_PR[1:] - x_PR[:-1]).T * (y_PR[:-1] + y_PR[1:])

    f1_score_list = 2 * TP / (len(real_score.T) + TP - TN)
    accuracy_list = (TP + TN) / len(real_score.T)
    specificity_list = TN / (TN + FP)

    max_index = np.argmax(f1_score_list)
    f1_score = f1_score_list[max_index]
    accuracy = accuracy_list[max_index]
    specificity = specificity_list[max_index]
    recall = recall_list[max_index]
    precision = precision_list[max_index]
    return [aupr[0, 0], auc[0, 0], f1_score, accuracy, recall, specificity, precision]

In [ ]:
import os
import gzip
import pickle
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.model_selection import ParameterGrid, KFold, train_test_split

from sgimc import SparseGroupIMCClassifier
from sgimc.utils import mc_split, get_submatrix


# ====== paths ======
PATH_TO_EXP = "/Users/sijianfan/projects/BiSSGL/datasets/simulations"
PATH_DATA = os.path.join(PATH_TO_EXP, "n_features")
PATH_OUTPUT = os.path.join(PATH_TO_EXP, "outputs_sgimc")
os.makedirs(PATH_OUTPUT, exist_ok=True)

# ====== simulation dataset grid ======
n_features_grid = np.arange(50, 501, 50)
n_repeats = 50
filename_template = "data_feature_{:03d}_rep_{:02d}.gz"

dataset_grid = []
for feature_id, n_features in enumerate(n_features_grid):
    for repeat_id in range(n_repeats):
        dataset_grid.append(
            {
                "feature_id": feature_id,
                "n_features": int(n_features),
                "repeat_id": int(repeat_id),
                "filename": os.path.join(
                    PATH_DATA,
                    filename_template.format(n_features, repeat_id),
                ),
            }
        )

print("Number of datasets:", len(dataset_grid))

Number of datasets: 500


In [10]:
# ====== choose one feature size for this notebook ======
target_n_features = 50
n_repeats = 50
filename_template = "data_feature_{:03d}_rep_{:02d}.gz"

dataset_grid = []
for repeat_id in range(n_repeats):
    dataset_grid.append(
        {
            "n_features": int(target_n_features),
            "repeat_id": int(repeat_id),
            "filename": os.path.join(
                PATH_DATA,
                filename_template.format(target_n_features, repeat_id),
            ),
        }
    )

print("Number of datasets:", len(dataset_grid))

Number of datasets: 50


In [29]:
results

[{'n_features': 50,
  'repeat_id': 0,
  'train_size': 0.05,
  'n_splits': 3,
  'C_lasso': 1.0,
  'C_group': 1.0,
  'C_ridge': 1.0,
  'rank': 25,
  'cv': 0,
  'val_score': [0.639754137811794,
   0.6726108961889903,
   0.6780072904009721,
   0.6397768819724383,
   0.7544055944055944,
   0.5238948062965407,
   0.6156597169380612],
  'val_d1': 49,
  'val_d2': 50,
  'test_score': [0.7416467895741015,
   0.7750970148251781,
   0.744114034761249,
   0.7314296875,
   0.7832395756616576,
   0.6799152423538943,
   0.7087114337568058],
  'test_d1': 48,
  'test_d2': 50},
 {'n_features': 50,
  'repeat_id': 0,
  'train_size': 0.05,
  'n_splits': 3,
  'C_lasso': 1.0,
  'C_group': 1.0,
  'C_ridge': 1.0,
  'rank': 25,
  'cv': 1,
  'val_score': [0.6282952336871537,
   0.6587425960893253,
   0.6699313786650031,
   0.5039609993906154,
   0.9991626349088203,
   0.0011336797354747285,
   0.5038945195195195],
  'val_d1': 49,
  'val_d2': 50,
  'test_score': [0.7416467895741015,
   0.7750970148251781,
   0.744

In [ ]:
# ====== parameter grids ======
grid_dataset = ParameterGrid(
    {
        "train_size": np.arange(0.05, 0.51, 0.05),
        "n_splits": [3],
    }
)

grid_model = ParameterGrid(
    {
        "C_lasso": [1.0, 1e-1, 1e-2],
        "C_group": [1.0, 1e-1, 1e-2],
        "C_ridge": [1.0, 1e-1, 1e-2],
        "rank": [25],
    }
)


# ====== random seed ======
random_state = 42
dvlp_size, test_size = 0.9, 0.1


# ====== main loop ======
results = []

for ds in tqdm(dataset_grid, desc="Datasets"):
    # load one dataset
    with gzip.open(ds["filename"], "rb") as fin:
        data = pickle.load(fin)

    U = data["X"]
    V = data["Y"]
    Y = data["R_noisy"].copy()
    Y_true = data["R"].copy()

    # split development / test
    ind_dvlp, ind_test = next(
        mc_split(
            Y,
            n_splits=1,
            random_state=random_state,
            train_size=dvlp_size,
            test_size=test_size,
        )
    )

    Y_test = get_submatrix(Y_true, ind_test)

    # loop over training settings
    for par_dtst in grid_dataset:
        # prepare the train dataset: take the specified share from the beginning of the index array
        ind_train_all, _ = train_test_split(
            ind_dvlp,
            shuffle=False,
            random_state=random_state,
            test_size=(1 - (par_dtst["train_size"] / dvlp_size)),
        )

        # loop over model settings
        for par_mdl in grid_model:
            C_lasso, C_group, C_ridge = (
                par_mdl["C_lasso"],
                par_mdl["C_group"],
                par_mdl["C_ridge"],
            )

            imc = SparseGroupIMCClassifier(
                par_mdl["rank"],
                n_threads=-1,
                random_state=42,
                C_lasso=C_lasso,
                C_group=C_group,
                C_ridge=C_ridge,
            )

            # fit on the whole development dataset
            Y_train = get_submatrix(Y, ind_train_all)
            Y_train[Y_train == 0] = -1

            imc.fit(U, V, Y_train)

            # get test score
            prob_full = imc.predict_proba(U, V)
            prob_test = get_submatrix(prob_full, ind_test)
            scores_test = get_metrics((Y_test.data + 1) / 2, prob_test.data)

            d1_test = int(sum(abs(imc.coef_W_).max(axis=1) > 0))
            d2_test = int(sum(abs(imc.coef_H_).max(axis=1) > 0))

            # k-fold CV
            splt = KFold(
                par_dtst["n_splits"],
                shuffle=True,
                random_state=random_state,
            )

            for cv, (ind_train, ind_valid) in enumerate(splt.split(ind_train_all)):
                ind_train = ind_train_all[ind_train]
                ind_valid = ind_train_all[ind_valid]

                Y_train = get_submatrix(Y, ind_train)
                Y_valid = get_submatrix(Y, ind_valid)

                Y_train[Y_train == 0] = -1

                imc = SparseGroupIMCClassifier(
                    par_mdl["rank"],
                    n_threads=-1,
                    random_state=42,
                    C_lasso=C_lasso,
                    C_group=C_group,
                    C_ridge=C_ridge,
                )
                imc.fit(U, V, Y_train)

                # validation score
                prob_full = imc.predict_proba(U, V)
                prob_valid = get_submatrix(prob_full, ind_valid)
                scores_valid = get_metrics((Y_valid.data + 1) / 2, prob_valid.data)

                d1_valid = int(sum(abs(imc.coef_W_).max(axis=1) > 0))
                d2_valid = int(sum(abs(imc.coef_H_).max(axis=1) > 0))

                results.append(
                    {
                        # "feature_id": ds["feature_id"],
                        "n_features": ds["n_features"],
                        "repeat_id": ds["repeat_id"],
                        "train_size": par_dtst["train_size"],
                        "n_splits": par_dtst["n_splits"],
                        "C_lasso": par_mdl["C_lasso"],
                        "C_group": par_mdl["C_group"],
                        "C_ridge": par_mdl["C_ridge"],
                        "rank": par_mdl["rank"],
                        "cv": cv,
                        "val_score": scores_valid,
                        "val_d1": d1_valid,
                        "val_d2": d2_valid,
                        "test_score": scores_test,
                        "test_d1": d1_test,
                        "test_d2": d2_test,
                    }
                )

df_results = pd.DataFrame(results)
df_results.head()

In [16]:
results

[{'n_features': 50,
  'repeat_id': 0,
  'train_size': 0.05,
  'n_splits': 3,
  'C_lasso': 1.0,
  'C_group': 1.0,
  'C_ridge': 1.0,
  'rank': 25,
  'cv': 0,
  'val_score': [0.639754137811794,
   0.6726108961889903,
   0.6780072904009721,
   0.6397768819724383,
   0.7544055944055944,
   0.5238948062965407,
   0.6156597169380612],
  'val_d1': 49,
  'val_d2': 50,
  'test_score': [0.7416467895741015,
   0.7750970148251781,
   0.744114034761249,
   0.7314296875,
   0.7832395756616576,
   0.6799152423538943,
   0.7087114337568058],
  'test_d1': 48,
  'test_d2': 50},
 {'n_features': 50,
  'repeat_id': 0,
  'train_size': 0.05,
  'n_splits': 3,
  'C_lasso': 1.0,
  'C_group': 1.0,
  'C_ridge': 1.0,
  'rank': 25,
  'cv': 1,
  'val_score': [0.6282952336871537,
   0.6587425960893253,
   0.6699313786650031,
   0.5039609993906154,
   0.9991626349088203,
   0.0011336797354747285,
   0.5038945195195195],
  'val_d1': 49,
  'val_d2': 50,
  'test_score': [0.7416467895741015,
   0.7750970148251781,
   0.744

In [32]:
results

[{'n_features': 50,
  'repeat_id': 0,
  'train_size': 0.05,
  'n_splits': 3,
  'C_lasso': 1.0,
  'C_group': 1.0,
  'C_ridge': 1.0,
  'rank': 25,
  'cv': 0,
  'val_score': [0.639754137811794,
   0.6726108961889903,
   0.6780072904009721,
   0.6397768819724383,
   0.7544055944055944,
   0.5238948062965407,
   0.6156597169380612],
  'val_d1': 49,
  'val_d2': 50,
  'test_score': [0.7416467895741015,
   0.7750970148251781,
   0.744114034761249,
   0.7314296875,
   0.7832395756616576,
   0.6799152423538943,
   0.7087114337568058],
  'test_d1': 48,
  'test_d2': 50},
 {'n_features': 50,
  'repeat_id': 0,
  'train_size': 0.05,
  'n_splits': 3,
  'C_lasso': 1.0,
  'C_group': 1.0,
  'C_ridge': 1.0,
  'rank': 25,
  'cv': 1,
  'val_score': [0.6282952336871537,
   0.6587425960893253,
   0.6699313786650031,
   0.5039609993906154,
   0.9991626349088203,
   0.0011336797354747285,
   0.5038945195195195],
  'val_d1': 49,
  'val_d2': 50,
  'test_score': [0.7416467895741015,
   0.7750970148251781,
   0.744

In [33]:
df_results = pd.DataFrame(results)
df_results

,n_features,repeat_id,train_size,n_splits,C_lasso,C_group,C_ridge,rank,cv,val_score,val_d1,val_d2,test_score,test_d1,test_d2
0,50,0,0.05,3,1.00,1.00,1.0000,25,0,"[0.639754137811794, 0.6726108961889903, 0.6780...",49,50,"[0.7416467895741015, 0.7750970148251781, 0.744...",48,50
1,50,0,0.05,3,1.00,1.00,1.0000,25,1,"[0.6282952336871537, 0.6587425960893253, 0.669...",49,50,"[0.7416467895741015, 0.7750970148251781, 0.744...",48,50
2,50,0,0.05,3,1.00,1.00,1.0000,25,2,"[0.6441588209683227, 0.6755138311329432, 0.674...",48,50,"[0.7416467895741015, 0.7750970148251781, 0.744...",48,50
3,50,0,0.05,3,1.00,1.00,0.0100,25,0,"[0.6649473774057917, 0.6987994536019798, 0.694...",49,50,"[0.757367826442017, 0.7889575968996492, 0.7537...",50,50
4,50,0,0.05,3,1.00,1.00,0.0100,25,1,"[0.6518584715068988, 0.6844587491445279, 0.682...",49,50,"[0.757367826442017, 0.7889575968996492, 0.7537...",50,50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1168,50,1,0.25,3,1.00,0.01,0.0001,25,1,"[0.936610877692807, 0.9343775034818151, 0.8538...",50,50,"[0.9892873969731408, 0.9889215512984737, 0.943...",50,50
1169,50,1,0.25,3,1.00,0.01,0.0001,25,2,"[0.9360215700153041, 0.9336036249919206, 0.854...",50,50,"[0.9892873969731408, 0.9889215512984737, 0.943...",50,50
1170,50,1,0.25,3,0.01,0.01,1.0000,25,0,"[0.9312648534759527, 0.9291860186972138, 0.851...",50,50,"[0.9873876503647686, 0.9869433248809145, 0.937...",50,50
1171,50,1,0.25,3,0.01,0.01,1.0000,25,1,"[0.9325974890109946, 0.9303292300613877, 0.849...",50,50,"[0.9873876503647686, 0.9869433248809145, 0.937...",50,50


In [34]:
outfile = os.path.join(
    PATH_OUTPUT,
    f"results_sgimc_feature_{target_n_features:03d}.pkl",
)

with open(outfile, "wb") as f:
    pickle.dump(df_results, f)

print("Saved to:", outfile)
print(df_results.shape)
df_results.head()

Saved to: /Users/sijianfan/projects/BiSSGL/datasets/simulations/outputs_sgimc/results_sgimc_feature_050.pkl
(1173, 15)


,n_features,repeat_id,train_size,n_splits,C_lasso,C_group,C_ridge,rank,cv,val_score,val_d1,val_d2,test_score,test_d1,test_d2
0,50,0,0.05,3,1.0,1.0,1.00,25,0,"[0.639754137811794, 0.6726108961889903, 0.6780...",49,50,"[0.7416467895741015, 0.7750970148251781, 0.744...",48,50
1,50,0,0.05,3,1.0,1.0,1.00,25,1,"[0.6282952336871537, 0.6587425960893253, 0.669...",49,50,"[0.7416467895741015, 0.7750970148251781, 0.744...",48,50
2,50,0,0.05,3,1.0,1.0,1.00,25,2,"[0.6441588209683227, 0.6755138311329432, 0.674...",48,50,"[0.7416467895741015, 0.7750970148251781, 0.744...",48,50
3,50,0,0.05,3,1.0,1.0,0.01,25,0,"[0.6649473774057917, 0.6987994536019798, 0.694...",49,50,"[0.757367826442017, 0.7889575968996492, 0.7537...",50,50
4,50,0,0.05,3,1.0,1.0,0.01,25,1,"[0.6518584715068988, 0.6844587491445279, 0.682...",49,50,"[0.757367826442017, 0.7889575968996492, 0.7537...",50,50


In [45]:
import numpy as np
import pandas as pd


def summarize_results(results):
    df = pd.DataFrame(results).copy()

    # make sure score columns are numpy arrays
    df["val_score"] = df["val_score"].apply(np.asarray)
    df["test_score"] = df["test_score"].apply(np.asarray)

    # ---------------------------------------------------
    # 1) group by train_size, repeat_id, and n_features,
    #    average val_score vector
    # ---------------------------------------------------
    summary_val = (
        df.groupby(["train_size", "repeat_id", "n_features"], as_index=False)
        .agg(
            val_score_mean=("val_score", lambda x: np.mean(np.stack(x), axis=0))
        )
    )

    # use the first element of averaged val_score vector for model selection
    summary_val["val_score_first"] = summary_val["val_score_mean"].apply(lambda x: x[0])

    # ---------------------------------------------------
    # 2) for each (train_size, repeat_id), keep only the
    #    n_features with the highest val_score_first
    # ---------------------------------------------------
    best_idx = summary_val.groupby(["train_size", "repeat_id", "n_features"])["val_score_first"].idxmax()
    best_combo = summary_val.loc[best_idx].reset_index(drop=True)

    # ---------------------------------------------------
    # 3) for each selected combo within each (train_size, repeat_id),
    #    calculate mean test_score vector across CV folds
    # ---------------------------------------------------
    df_best = df.merge(
        best_combo[["train_size", "repeat_id", "n_features"]],
        on=["train_size", "repeat_id", "n_features"],
        how="inner"
    )

    summary_test = (
        df_best.groupby(["train_size", "repeat_id", "n_features"], as_index=False)
        .agg(
            test_score_mean=("test_score", lambda x: np.mean(np.stack(x), axis=0))
        )
    )

    # ---------------------------------------------------
    # 4) across repeats, calculate average and std of the
    #    selected test_score_mean for each train_size
    # ---------------------------------------------------
    final_summary = (
        summary_test.groupby("train_size", as_index=False)
        .agg(
            test_score_avg=("test_score_mean", lambda x: np.mean(np.stack(x), axis=0)),
            test_score_std=("test_score_mean", lambda x: np.std(np.stack(x), axis=0, ddof=1)),
        )
    )

    return summary_val, best_combo, summary_test, final_summary

In [46]:
summary_val, best_combo, summary_test, final_summary = summarize_results(results)

/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/numpy/core/_methods.py:265: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/numpy/core/_methods.py:254: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


In [47]:
METRIC_NAMES = ["aupr", "auc", "f1", "acc", "recall", "specificity", "precision"]

def expand_vector_column(df, col, prefix, metric_names=METRIC_NAMES):
    out = df.copy()
    vec_df = pd.DataFrame(out[col].tolist(), index=out.index)
    vec_df.columns = [f"{prefix}_{m}" for m in metric_names[:vec_df.shape[1]]]
    out = pd.concat([out.drop(columns=[col]), vec_df], axis=1)
    return out

summary_val_readable = expand_vector_column(summary_val, "val_score_mean", "val_mean").round(4)
summary_test_readable = expand_vector_column(summary_test, "test_score_mean", "test_mean").round(4)

final_summary_readable = expand_vector_column(final_summary, "test_score_avg", "test_avg")
final_summary_readable = expand_vector_column(final_summary_readable, "test_score_std", "test_std")
final_summary_readable = final_summary_readable.round(4)

final_summary_compact = final_summary_readable[["train_size"]].copy()
for m in METRIC_NAMES:
    avg_col = f"test_avg_{m}"
    std_col = f"test_std_{m}"
    if avg_col in final_summary_readable.columns:
        final_summary_compact[m] = final_summary_readable.apply(
            lambda r: f"{r[avg_col]:.4f} ± {r[std_col]:.4f}", axis=1
        )

display(summary_val_readable.head())
display(summary_test_readable.head())
display(final_summary_readable)
display(final_summary_compact)

,train_size,repeat_id,n_features,val_score_first,val_mean_aupr,val_mean_auc,val_mean_f1,val_mean_acc,val_mean_recall,val_mean_specificity,val_mean_precision
0,0.05,0,50,0.6673,0.6673,0.6990,0.6911,0.6501,0.7784,0.5204,0.6225
1,0.05,1,50,0.7316,0.7316,0.7581,0.7269,0.6976,0.8008,0.5935,0.6659
2,0.10,0,50,0.7909,0.7909,0.8087,0.7594,0.7394,0.8177,0.6603,0.7090
3,0.10,1,50,0.8505,0.8505,0.8588,0.7913,0.7783,0.8405,0.7160,0.7476
4,0.15,0,50,0.8583,0.8583,0.8648,0.7967,0.7849,0.8414,0.7282,0.7566


,train_size,repeat_id,n_features,test_mean_aupr,test_mean_auc,test_mean_f1,test_mean_acc,test_mean_recall,test_mean_specificity,test_mean_precision
0,0.05,0,50,0.7566,0.7869,0.7493,0.7349,0.7945,0.6757,0.7090
1,0.05,1,50,0.8322,0.8514,0.7950,0.7857,0.8325,0.7391,0.7609
2,0.10,0,50,0.8920,0.8988,0.8335,0.8291,0.8581,0.8003,0.8104
3,0.10,1,50,0.9365,0.9379,0.8698,0.8671,0.8895,0.8448,0.8511
4,0.15,0,50,0.9452,0.9451,0.8785,0.8770,0.8919,0.8622,0.8655


,train_size,test_avg_aupr,test_avg_auc,test_avg_f1,test_avg_acc,test_avg_recall,test_avg_specificity,test_avg_precision,test_std_aupr,test_std_auc,test_std_f1,test_std_acc,test_std_recall,test_std_specificity,test_std_precision
0,0.05,0.7944,0.8192,0.7721,0.7603,0.8135,0.7074,0.7350,0.0534,0.0456,0.0323,0.0359,0.0268,0.0449,0.0367
1,0.10,0.9142,0.9184,0.8517,0.8481,0.8738,0.8226,0.8307,0.0314,0.0276,0.0257,0.0269,0.0222,0.0315,0.0288
2,0.15,0.9571,0.9568,0.8924,0.8912,0.9045,0.8779,0.8806,0.0169,0.0165,0.0197,0.0201,0.0179,0.0222,0.0213
3,0.20,0.9753,0.9744,0.9165,0.9159,0.9249,0.9071,0.9083,0.0089,0.0092,0.0140,0.0141,0.0136,0.0146,0.0144
4,0.25,0.9843,0.9837,0.9327,0.9326,0.9373,0.9279,0.9282,0.0059,0.0062,0.0121,0.0120,0.0130,0.0110,0.0112
5,0.30,0.9869,0.9863,0.9377,0.9374,0.9448,0.9300,0.9307,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.35,0.9909,0.9906,0.9482,0.9481,0.9523,0.9439,0.9441,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.40,0.9935,0.9933,0.9560,0.9559,0.9592,0.9527,0.9527,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0.45,0.9952,0.9951,0.9619,0.9620,0.9619,0.9622,0.9619,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,0.50,0.9963,0.9962,0.9662,0.9663,0.9669,0.9657,0.9655,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,train_size,aupr,auc,f1,acc,recall,specificity,precision
0,0.05,0.7944 ± 0.0534,0.8192 ± 0.0456,0.7721 ± 0.0323,0.7603 ± 0.0359,0.8135 ± 0.0268,0.7074 ± 0.0449,0.7350 ± 0.0367
1,0.10,0.9142 ± 0.0314,0.9184 ± 0.0276,0.8517 ± 0.0257,0.8481 ± 0.0269,0.8738 ± 0.0222,0.8226 ± 0.0315,0.8307 ± 0.0288
2,0.15,0.9571 ± 0.0169,0.9568 ± 0.0165,0.8924 ± 0.0197,0.8912 ± 0.0201,0.9045 ± 0.0179,0.8779 ± 0.0222,0.8806 ± 0.0213
3,0.20,0.9753 ± 0.0089,0.9744 ± 0.0092,0.9165 ± 0.0140,0.9159 ± 0.0141,0.9249 ± 0.0136,0.9071 ± 0.0146,0.9083 ± 0.0144
4,0.25,0.9843 ± 0.0059,0.9837 ± 0.0062,0.9327 ± 0.0121,0.9326 ± 0.0120,0.9373 ± 0.0130,0.9279 ± 0.0110,0.9282 ± 0.0112
5,0.30,0.9869 ± nan,0.9863 ± nan,0.9377 ± nan,0.9374 ± nan,0.9448 ± nan,0.9300 ± nan,0.9307 ± nan
6,0.35,0.9909 ± nan,0.9906 ± nan,0.9482 ± nan,0.9481 ± nan,0.9523 ± nan,0.9439 ± nan,0.9441 ± nan
7,0.40,0.9935 ± nan,0.9933 ± nan,0.9560 ± nan,0.9559 ± nan,0.9592 ± nan,0.9527 ± nan,0.9527 ± nan
8,0.45,0.9952 ± nan,0.9951 ± nan,0.9619 ± nan,0.9620 ± nan,0.9619 ± nan,0.9622 ± nan,0.9619 ± nan
9,0.50,0.9963 ± nan,0.9962 ± nan,0.9662 ± nan,0.9663 ± nan,0.9669 ± nan,0.9657 ± nan,0.9655 ± nan


In [44]:
print(final_summary_compact)

   train_size             aupr              auc               f1  \
0        0.05  0.7944 ± 0.0534  0.8192 ± 0.0456  0.7721 ± 0.0323   
1        0.10  0.9142 ± 0.0314  0.9184 ± 0.0276  0.8517 ± 0.0257   
2        0.15  0.9571 ± 0.0169  0.9568 ± 0.0165  0.8924 ± 0.0197   
3        0.20  0.9753 ± 0.0089  0.9744 ± 0.0092  0.9165 ± 0.0140   
4        0.25  0.9843 ± 0.0059  0.9837 ± 0.0062  0.9327 ± 0.0121   
5        0.30     0.9869 ± nan     0.9863 ± nan     0.9377 ± nan   
6        0.35     0.9909 ± nan     0.9906 ± nan     0.9482 ± nan   
7        0.40     0.9935 ± nan     0.9933 ± nan     0.9560 ± nan   
8        0.45     0.9952 ± nan     0.9951 ± nan     0.9619 ± nan   
9        0.50     0.9963 ± nan     0.9962 ± nan     0.9662 ± nan   

               acc           recall      specificity        precision  
0  0.7603 ± 0.0359  0.8135 ± 0.0268  0.7074 ± 0.0449  0.7350 ± 0.0367  
1  0.8481 ± 0.0269  0.8738 ± 0.0222  0.8226 ± 0.0315  0.8307 ± 0.0288  
2  0.8912 ± 0.0201  0.9045 ± 0.0179

In [72]:
from collections import Counter

count = Counter(x.get('train_size') for x in results)

In [113]:
{k: v / 3  for k, v in count.items()}

{0.05: 54.0,
 0.1: 54.0,
 0.15000000000000002: 54.0,
 0.2: 54.0,
 0.25: 40.0,
 0.30000000000000004: 27.0,
 0.35000000000000003: 27.0,
 0.4: 27.0,
 0.45: 27.0,
 0.5: 27.0}

In [151]:
matching = [x for x in results if x.get('train_size') == 0.05]

In [152]:
pd.DataFrame({tuple(x.get('test_score')) for x in matching})

,0,1,2,3,4,5,6
0,0.761940,0.790692,0.750424,0.736555,0.794396,0.679043,0.711064
1,0.746277,0.777552,0.741884,0.725992,0.789821,0.662527,0.699434
2,0.828657,0.850134,0.797891,0.791219,0.825860,0.756713,0.771755
3,0.759194,0.788719,0.749292,0.732352,0.802216,0.662886,0.702919
4,0.834577,0.852403,0.793112,0.781109,0.840780,0.721673,0.750559
5,0.823929,0.843813,0.787140,0.776281,0.828929,0.723841,0.749363
6,0.823974,0.846404,0.795592,0.788844,0.823481,0.754343,0.769531
7,0.823869,0.843787,0.787191,0.776547,0.828193,0.725104,0.750057
8,0.834345,0.852242,0.793163,0.780750,0.842424,0.719318,0.749345
9,0.748163,0.780924,0.747954,0.737477,0.781281,0.693922,0.717354


In [78]:
display(count)

Counter({0.05: 162,
         0.1: 162,
         0.15000000000000002: 162,
         0.2: 162,
         0.25: 120,
         0.30000000000000004: 81,
         0.35000000000000003: 81,
         0.4: 81,
         0.45: 81,
         0.5: 81})